# SeedDance 火山引擎 API 连通性测试

**运行前准备：**
在 Colab 左侧边栏 → 🔑 Secrets，添加以下 secret：
- `VOLCENGINE_API_KEY` — 火山引擎 Ark API key（UUID 格式）
- `YUNWU_API_KEY` — 云雾 AI key（图像生成用）
- `OPENROUTER_API_KEY` — OpenRouter key（对话模型用）

## Cell 1 — 克隆仓库 & 安装依赖

In [ ]:
!pip install uv -q
!git clone https://github.com/RoboRabbit666/ViMax.git /content/ViMax
%cd /content/ViMax
!uv sync --no-dev 2>&1 | tail -5
print("环境准备完成")

## Cell 2 — 读取 API Key

In [ ]:
from google.colab import userdata
VOLCENGINE_API_KEY = userdata.get('VOLCENGINE_API_KEY')
YUNWU_API_KEY = userdata.get('YUNWU_API_KEY')
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
assert VOLCENGINE_API_KEY, "缺少 VOLCENGINE_API_KEY"
assert YUNWU_API_KEY, "缺少 YUNWU_API_KEY"
assert OPENROUTER_API_KEY, "缺少 OPENROUTER_API_KEY"
print("Key 读取成功")

## Cell 3 — API 连通性测试（t2v + ff2v）

In [ ]:
import os
os.environ['VOLCENGINE_API_KEY'] = VOLCENGINE_API_KEY
!cd /content/ViMax && uv run python tests/test_seedance_volcengine.py

## Cell 4 — 写配置文件（SeedDance 2.0）

In [ ]:
import yaml, os

cfg = {
    'chat_model': {
        'init_args': {
            'model': 'google/gemini-2.5-flash-lite-preview-09-2025',
            'model_provider': 'openai',
            'api_key': OPENROUTER_API_KEY,
            'base_url': 'https://openrouter.ai/api/v1'
        },
        'max_requests_per_minute': 500,
        'max_requests_per_day': 2000,
    },
    'image_generator': {
        'class_path': 'tools.ImageGeneratorDoubaoSeedreamYunwuAPI',
        'init_args': {'api_key': YUNWU_API_KEY},
        'max_requests_per_minute': 10,
        'max_requests_per_day': 500,
    },
    'video_generator': {
        'class_path': 'tools.VideoGeneratorDoubaoSeedanceVolcengineAPI',
        'init_args': {
            'api_key': VOLCENGINE_API_KEY,
            't2v_model': 'ep-20260416124751-x4tfn',
            'ff2v_model': 'ep-20260416124751-x4tfn',
            'flf2v_model': 'ep-20260416124751-x4tfn',
        },
        'max_requests_per_minute': 2,
        'max_requests_per_day': 50,
    },
}

os.makedirs('/content/vimax_outputs', exist_ok=True)
with open('/content/ViMax/configs/idea2video.yaml', 'w') as f:
    yaml.dump({**cfg, 'working_dir': '/content/vimax_outputs/idea2video'}, f, allow_unicode=True)
print("配置写入完成")

## Cell 5 — 运行完整 ViMax 流水线

In [ ]:
!cd /content/ViMax && uv run python main_idea2video.py

## Cell 6 — 打包下载输出

In [ ]:
!zip -r /content/vimax_outputs.zip /content/vimax_outputs
from google.colab import files
files.download('/content/vimax_outputs.zip')